# 1. Imports and Data Loading

In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.1 MB/s eta 0:00:00


In [ ]:
# Load data.
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/airbnb_review_text_analytics')

import pandas as pd

df = pd.read_csv("data/processed/segment_df_with_topics_final.csv")

print(df.head())

Mounted at /content/drive
   segment_id            review_id           listing_id  \
0           1            463567767             10412695   
1           2            463567767             10412695   
2           3            463567767             10412695   
3           5            463567767             10412695   
4           9  1515313053468489649  1467062721079452795   

                                   segment_text                 segment_norm  \
0           The pub downstairs has awesome food  pub downstairs awesome food   
1  and seriously the best ceasar I've ever had.     seriously good ceasar ve   
2                The roof top patio was amazing           roof patio amazing   
3                   The neighborhood is perfect         neighborhood perfect   
4        Came back to the city to visit friends       come city visit friend   

   main_topic_id main_topic_label  topic_id                 topic_label  
0              7         Location        28  Nearby Restaurants 

# 2. VADER Sentiment Scoring

In [ ]:
# Compute VADER scores for each segment.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

vader_scores = df['segment_text'].apply(
    lambda text: analyzer.polarity_scores(str(text))
)

df['compound'] = vader_scores.apply(lambda x: x['compound'])

# Convert compound to a 0–100 satisfaction index.
df['satisfaction_0_100'] = (df['compound'] + 1) / 2 * 100

df.head()

,segment_id,review_id,listing_id,segment_text,segment_norm,main_topic_id,main_topic_label,topic_id,topic_label,compound,satisfaction_0_100
0,1,463567767,10412695,The pub downstairs has awesome food,pub downstairs awesome food,7,Location,28,Nearby Restaurants & Shops,0.6249,81.245
1,2,463567767,10412695,and seriously the best ceasar I've ever had.,seriously good ceasar ve,6,Experience,23,Overall Experience,0.5423,77.115
2,3,463567767,10412695,The roof top patio was amazing,roof patio amazing,6,Experience,25,View Experience,0.6808,84.040
3,5,463567767,10412695,The neighborhood is perfect,neighborhood perfect,7,Location,27,Location Convenience,0.5719,78.595
4,9,1515313053468489649,1467062721079452795,Came back to the city to visit friends,come city visit friend,7,Location,27,Location Convenience,0.4767,73.835


In [ ]:
# Classify each segment based on compound score thresholds.
def sentiment_label(compound):
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label'] = df['compound'].apply(sentiment_label)

print(df['sentiment_label'].value_counts())

sentiment_label
positive    20253
neutral      7874
negative     1141
Name: count, dtype: int64


In [ ]:
# Drop unnecessary columns and save the final segment-level output.
df = df.drop(columns=["segment_norm", "compound"])

os.makedirs("data/final", exist_ok=True)

df.to_csv("data/final/segment_df_with_topics_and_vader.csv", index=False)

df.head()

,segment_id,review_id,listing_id,segment_text,main_topic_id,main_topic_label,topic_id,topic_label,satisfaction_0_100,sentiment_label
0,1,463567767,10412695,The pub downstairs has awesome food,7,Location,28,Nearby Restaurants & Shops,81.245,positive
1,2,463567767,10412695,and seriously the best ceasar I've ever had.,6,Experience,23,Overall Experience,77.115,positive
2,3,463567767,10412695,The roof top patio was amazing,6,Experience,25,View Experience,84.040,positive
3,5,463567767,10412695,The neighborhood is perfect,7,Location,27,Location Convenience,78.595,positive
4,9,1515313053468489649,1467062721079452795,Came back to the city to visit friends,7,Location,27,Location Convenience,73.835,positive


In [ ]:
# Aggregate by main topic.
# Mean satisfaction per main topic.
main_topic_satisfaction = df.groupby('main_topic_label')['satisfaction_0_100'].mean().rename('mean_satisfaction')

# Positive, negative, and neutral rates per main topic.
main_topic_sentiment_counts = (
    df.groupby(['main_topic_label', 'sentiment_label']).size()
    .unstack(fill_value=0)
)
main_topic_sentiment_rates = main_topic_sentiment_counts.div(main_topic_sentiment_counts.sum(axis=1), axis=0)
main_topic_sentiment_rates.columns = [f'{c}_rate' for c in main_topic_sentiment_rates.columns]

main_topic_metrics = main_topic_satisfaction.to_frame().join(main_topic_sentiment_rates)
main_topic_metrics = main_topic_metrics.reset_index()

main_topic_metrics

,main_topic_label,mean_satisfaction,negative_rate,neutral_rate,positive_rate
0,Accuracy,68.929094,0.018304,0.305395,0.676301
1,Amenities,61.948393,0.089164,0.398641,0.512195
2,Check-in,65.317662,0.070696,0.305587,0.623717
3,Cleanliness,72.334092,0.035954,0.174976,0.789070
4,Comfort,67.654039,0.056836,0.297442,0.645722
5,Communication,74.799732,0.022128,0.167597,0.810275
6,Experience,71.764940,0.022395,0.208432,0.769173
7,Location,65.607695,0.044842,0.391919,0.563239
8,Value,71.615903,0.046729,0.193146,0.760125


In [ ]:
# Aggregate by detailed topic.
# Mean satisfaction per detailed topic.
topic_satisfaction = (
    df.groupby(["main_topic_label", "topic_label"])["satisfaction_0_100"]
    .mean()
    .rename("mean_satisfaction")
)

# Positive, negative, and neutral rates per detailed topic.
topic_sentiment_counts = (
    df.groupby(["main_topic_label", "topic_label", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)

topic_sentiment_rates = topic_sentiment_counts.div(
    topic_sentiment_counts.sum(axis=1),
    axis=0
)

topic_sentiment_rates.columns = [
    f"{c}_rate" for c in topic_sentiment_rates.columns
]

# Combine satisfaction and sentiment rate metrics.
topic_metrics = topic_satisfaction.to_frame().join(topic_sentiment_rates)
topic_metrics = topic_metrics.reset_index()

topic_metrics

,main_topic_label,topic_label,mean_satisfaction,negative_rate,neutral_rate,positive_rate
0,Accuracy,Apartment Expectation,72.366252,0.016973,0.216407,0.766620
1,Accuracy,Photo Accuracy,61.587492,0.021148,0.495468,0.483384
2,Amenities,Coffee & Breakfast,65.683794,0.041958,0.370629,0.587413
3,Amenities,General Amenities,66.815800,0.012000,0.400000,0.588000
4,Amenities,Kitchen Facilities,60.668345,0.076014,0.429054,0.494932
5,Amenities,Laundry & Towels,57.940692,0.107692,0.507692,0.384615
6,Amenities,Parking,63.328253,0.098107,0.359725,0.542169
7,Amenities,Temperature Control,60.606374,0.118321,0.370229,0.511450
8,Amenities,Wi-Fi & TV,58.655829,0.195122,0.321951,0.482927
9,Check-in,Check-in,65.317662,0.070696,0.305587,0.623717
